# Planning Within the Onboard Recorder

## Introduction

Science data goes onto the onboard recorder as it is collected and comes off only
during ground contacts. An instrument that generates data quickly can fill the
recorder long before the next contact, and when it does the data is lost and, since
the recorder's fill fraction is a fault threshold, the spacecraft goes to safe mode.

COASTSim's schedulers plan within the recorder. Every plan keeps the recorder below
`recorder.planning_limit`, a fraction of its capacity that defaults to the red
threshold:

* Each instrument's `DataGeneration` says what it generates: science data while
  collecting (a rate, plus a fixed amount per observation) and housekeeping data
  at all times. A target can set the science data each instrument generates while
  observing it, since a bright source generates more than a faint one.
* The reserved ground contacts cut the horizon into segments. The scheduler keeps
  the volume at the end of every segment, just before the next contact, within the
  limit. Each contact removes what its rate and length allow, and no more than is
  there, so what it cannot remove carries over.
* A snapshot that does not fit before a contact is placed after one. Snapshots are
  never shortened to fit, and requests are placed in merit order, so when data runs
  short it is the lowest-merit science that waits, whatever its data rate.

This notebook plans a day for a spacecraft whose recorder holds a few hours of
science, with a handful of bright targets that each fill a sixth of it in one visit.
It plans the day with and without the limit, executes each plan, and shows what
happens to the recorder and where the bright visits go.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import EarthLimbConstraint, SunConstraint, TLEEphemeris

from conops import DITL, RollingHorizonDITL
from conops.common import ACSMode, ObsType
from conops.config import (
    AttitudeControlSystem,
    Battery,
    Constraint,
    DataGeneration,
    GroundStationRegistry,
    HousekeepingData,
    MissionConfig,
    OnboardRecorder,
    Payload,
    RadiatorConfiguration,
    ScienceData,
    SolarPanelSet,
    SpacecraftBus,
    StarTrackerConfiguration,
    Telescope,
    TelescopeConfig,
    TelescopeType,
)
from conops.schedulers import LocalSearchScheduler, PriorityScheduler
from conops.targets import Pointing

## Step 2: Spacecraft Configuration and Ephemeris

A spacecraft with one imaging telescope, the default ground stations (S-band at
10 Mbps, so a typical 8-minute contact downlinks about 5 Gb) and a 32 Gb recorder.
Power and pointing are kept simple so that the recorder is the only thing that
limits the day: a large battery, no solar panels, star trackers or radiators, and
Sun and Earth-limb constraints only, as in the scheduler tests.

The imager generates 0.5 Mbps of science data while collecting and 50 kbps of
housekeeping at all times; the bus adds another 50 kbps of housekeeping, so the
recorder gains 0.36 Gb an hour whatever the spacecraft does. The recorder's red
threshold is 90%, so by default the schedulers plan to keep it below 28.8 Gb.

The day starts at 05:00 UTC. With these stations the first contact comes three
hours in, nine more follow over the next ten hours, and the last eleven hours of
the day have none, so the recorder has to carry the evening's science overnight.

In [ ]:
IMAGER = "Imager"
CAPACITY_GB = 32.0

config = MissionConfig(
    ground_stations=GroundStationRegistry.default(),
    recorder=OnboardRecorder(capacity_gb=CAPACITY_GB),
    constraint=Constraint(
        sun_constraint=SunConstraint(min_angle=45),
        earth_constraint=EarthLimbConstraint(min_angle=10),
    ),
    solar_panel=SolarPanelSet(panels=[]),
    battery=Battery(watthour=100_000.0),
    spacecraft_bus=SpacecraftBus(
        attitude_control=AttitudeControlSystem(
            max_slew_rate=1.0, slew_acceleration=0.5, settle_time=30.0
        ),
        star_trackers=StarTrackerConfiguration(
            star_trackers=[], min_functional_trackers=0, modes_require_lock=[]
        ),
        radiators=RadiatorConfiguration(radiators=[]),
        data_generation=HousekeepingData(rate_gbps=0.00005),
    ),
    payload=Payload(
        instruments=[
            Telescope(
                name=IMAGER,
                optics=TelescopeConfig(
                    aperture_m=0.5,
                    focal_length_m=3.5,
                    telescope_type=TelescopeType.RITCHEY_CHRETIEN,
                    tube_length_m=1.0,
                ),
                data_generation=DataGeneration(
                    science=ScienceData(rate_gbps=0.0005),
                    housekeeping=HousekeepingData(rate_gbps=0.00005),
                ),
            )
        ]
    ),
)
config.random_seed = 1234

begin = datetime(2025, 11, 1, 5, tzinfo=timezone.utc)
end = begin + timedelta(days=1)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=60, tle="example.tle")
config.constraint.ephem = ephemeris

print(
    f"Planning limit: {config.recorder.planning_limit_gb:.0f} Gb of {CAPACITY_GB:.0f}"
)

## Step 3: Create the Targets

Sixty targets at random positions. Fifty-two are faint: each needs an hour of
exposure in 20-minute snapshots at the imager's own rate, 0.6 Gb a snapshot, and
together they ask for far more than a day. Eight are bright: each needs a single
20-minute visit, and observing one generates 4.2 Mbps in the imager, so that visit
is 5 Gb, a sixth of the recorder. They are also the highest-merit targets, so a
scheduler that ranks by merit alone wants to observe them first.

A target's `science_data` sets what each instrument generates while observing it,
by instrument name. Targets without it generate the instrument's own rate.

In [ ]:
BRIGHT_RATE = 0.0042  # Gbps
FAINT_RATE = 0.0005  # the imager's own rate
SNAPSHOT = 1200  # s
BRIGHT_COUNT = 8


def make_targets(seed: int) -> list[Pointing]:
    rng = np.random.default_rng(seed)
    targets = []
    for i in range(60):
        bright = i < BRIGHT_COUNT
        target = Pointing(
            config=config,
            ra=rng.uniform(0, 360),
            dec=rng.uniform(-60, 60),
            obsid=(20000 if bright else 10000) + i,
            name=f"{'Bright' if bright else 'Faint'} {i}",
            merit=rng.uniform(70, 90) if bright else rng.uniform(20, 70),
            ss_min=SNAPSHOT,
            ss_max=SNAPSHOT,
            science_data={IMAGER: ScienceData(rate_gbps=BRIGHT_RATE)} if bright else {},
        )
        target.exptime = SNAPSHOT if bright else 3 * SNAPSHOT
        targets.append(target)
    return targets


targets = make_targets(7)
bright_ids = {t.obsid for t in targets if t.obsid >= 20000}
print(
    f"A bright snapshot generates {config.science_generated(SNAPSHOT, targets[0].science_data):.1f} Gb, "
    f"a faint one {config.science_generated(SNAPSHOT):.1f} Gb"
)

## Step 4: Plan the Day With and Without the Limit

`PriorityScheduler` places requests in merit order, each at the earliest time it
fits. With the limit, "fits" includes the recorder: a snapshot goes only where
every peak from there to the end of the day stays within 28.8 Gb.

To plan without regard to the recorder, set `planning_limit=None`.

In [ ]:
def plan_day(limit, label):
    config.recorder.planning_limit = limit
    # A simulation leaves its data on the configured recorder, so empty it
    # before planning: the scheduler starts from what is on it.
    config.recorder.current_volume_gb = 0.0
    scheduler = PriorityScheduler(config, targets, begin, end)
    plan = scheduler.schedule()
    science = [e for e in plan if e.obstype == ObsType.AT]
    bright = [e for e in science if e.obsid in bright_ids]
    print(
        f"{label:<14} {len(science):3d} snapshots, {len(bright):2d} of bright targets, "
        f"{sum(e.exposure for e in science) / 3600:5.2f} h of science"
    )
    return scheduler, plan


limited_scheduler, limited_plan = plan_day(0.9, "With limit")
unlimited_scheduler, unlimited_plan = plan_day(None, "Without limit")
config.recorder.planning_limit = 0.9

## Step 5: What the Scheduler Expects the Recorder to Do

The scheduler's ledger of the recorder: for each segment of the day, the contact
that starts it and what it downlinks, the science volume the plan puts in the
segment, and the peak volume just before the next contact. The plan keeps every
peak under 28.8 Gb; segments that reach it are where the recorder, not the sky,
decided what was observed. The bright visits show up as segments with 5 Gb or
more of science: as many as the recorder can hold before the first contact, then
one wherever a contact has made room.

In [ ]:
ledger = limited_scheduler._ledger
peaks = ledger.peaks(limited_scheduler._volumes)
print(f"{'segment':>7} {'starts at':>9} {'downlink':>8} {'science':>8} {'peak':>7}")
for k in range(ledger.segments):
    start = (ledger.bounds[k] - begin.timestamp()) / 3600
    print(
        f"{k:7d} {start:8.2f}h {ledger.downlink_gb[k]:7.1f} Gb "
        f"{limited_scheduler._volumes[k]:7.1f} Gb {peaks[k]:6.1f} Gb"
        + ("  <- at the limit" if peaks[k] > ledger.limit_gb - 1.5 else "")
    )

## Step 6: Execute Both Plans

Each plan is executed by a `DITL`, which generates the data the configuration says
and downlinks during the contacts. `validate_plan_matches_execution()` is empty
when the simulation did exactly what the plan said. A simulation works on the
configured recorder itself, so each run starts by emptying it.

In [ ]:
def execute(plan, initial_gb=0.0) -> DITL:
    config.recorder.current_volume_gb = initial_gb
    ditl = DITL(config=config, plan=plan, ephem=ephemeris, begin=begin, end=end)
    ditl.step_size = 60
    ditl.calc()
    return ditl


limited = execute(limited_plan)
unlimited = execute(unlimited_plan)


def report(label, run, initial_gb=0.0):
    volume = np.asarray(run.recorder_volume_gb)
    safe = ACSMode.SAFE in run.mode
    # What was generated and neither downlinked nor still on the recorder.
    lost = (
        run.data_generated_gb[-1]
        - run.data_downlinked_gb[-1]
        - (volume[-1] - initial_gb)
    )
    print(
        f"{label:<14} peak {volume.max():5.1f} Gb, "
        f"{run.data_generated_gb[-1]:5.1f} Gb generated, "
        f"{run.data_downlinked_gb[-1]:5.1f} Gb downlinked, "
        f"{max(lost, 0.0):4.1f} Gb lost, "
        f"{len(run.validate_plan_matches_execution()):3d} mismatches"
        + (", went to SAFE mode" if safe else "")
    )


report("With limit", limited)
report("Without limit", unlimited)

## Step 7: The Recorder Through the Day

Recorder volume for each plan, with the contacts shaded. The limited plan rises to
the planning limit before contacts and no further, and carries the evening's science
through the eleven hours without contacts. The unlimited plan observes the bright
targets back to back from the start and runs the recorder to the red threshold
before the first contact, at which point fault management puts the spacecraft into
safe mode and the rest of the day is lost.

In [ ]:
def plot_recorder(ax, run, label, color):
    hours = (np.asarray(run.utime) - begin.timestamp()) / 3600
    ax.plot(hours, run.recorder_volume_gb, color=color, label=label)
    modes = np.asarray(run.mode)
    if ACSMode.SAFE in modes:
        safe = modes == ACSMode.SAFE
        ax.fill_between(
            hours,
            0,
            CAPACITY_GB,
            where=safe,
            color="black",
            alpha=0.08,
            label="safe mode",
        )


def shade_contacts(ax, plan):
    first = True
    for e in plan:
        if e.obstype == ObsType.GSP and e.contact_begin is not None:
            b = (e.contact_begin - begin.timestamp()) / 3600
            f = (e.contact_end - begin.timestamp()) / 3600
            ax.axvspan(
                b, f, color="tab:orange", alpha=0.25, label="contact" if first else None
            )
            first = False


fig, ax = plt.subplots(figsize=(10, 4))
plot_recorder(ax, limited, "With limit", "tab:blue")
plot_recorder(ax, unlimited, "Without limit", "tab:red")
shade_contacts(ax, limited_plan)
ax.axhline(
    config.recorder.planning_limit_gb,
    color="tab:blue",
    ls="--",
    lw=1,
    label="planning limit (the red threshold, 28.8 Gb)",
)
ax.axhline(CAPACITY_GB, color="black", lw=0.8, label="capacity")
ax.set_xlim(0, 24)
ax.set_ylim(0, CAPACITY_GB * 1.05)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Recorder volume (Gb)")
ax.legend(ncol=3, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## Step 8: Who Got Observed

Exposure planned for each target, highest merit first, in each plan. Both plans
observe all eight bright targets: with the limit they still come first, since they
have the highest merit, but only as many as the recorder can hold until the first
contact go in the morning, and the rest wait for a contact to make 5 Gb of room.
The faint targets fill in around them.

In [ ]:
def planned_seconds(plan):
    seconds = {}
    for e in plan:
        if e.obstype == ObsType.AT:
            seconds[e.obsid] = seconds.get(e.obsid, 0.0) + e.exposure
    return seconds


by_merit = sorted(targets, key=lambda t: t.merit, reverse=True)
x = np.arange(len(by_merit))
fig, ax = plt.subplots(figsize=(10, 3.5))
for offset, (label, plan) in zip(
    (-0.2, 0.2), (("Without limit", unlimited_plan), ("With limit", limited_plan))
):
    seconds = planned_seconds(plan)
    ax.bar(
        x + offset, [seconds.get(t.obsid, 0) / 60 for t in by_merit], 0.4, label=label
    )
ax.set_xticks(
    x[:BRIGHT_COUNT],
    [t.name for t in by_merit[:BRIGHT_COUNT]],
    rotation=45,
    ha="right",
    fontsize=8,
)
ax.set_xlabel("Targets, highest merit first (the bright targets labelled)")
ax.set_ylabel("Planned exposure (min)")
ax.legend()
fig.tight_layout()
plt.show()

## Step 9: Local Search Within the Limit

`LocalSearchScheduler` starts from the priority-first plan and tries changes to its
order of snapshots, keeping those that raise merit-weighted science: each snapshot's
merit times its science seconds. Every plan it considers is checked against the
recorder too, so what it searches for is the best use of the day's downlink.

The priority-first plan left idle time where the recorder was full: it placed the
bright visits first, because they outrank everything else, and the faint targets
could only use what room was left. Local search fills most of that idle time. It
moves faint snapshots into the gaps, and it gives up one bright visit, whose 5 Gb
buys eight faint snapshots that together are worth more merit-weighted science.
Whether such a trade is right depends on the merits: a higher merit for the bright
targets, or a per-observation amount that made faint snapshots dearer, would keep
it; the search follows the merits it is given.

In [ ]:
config.recorder.current_volume_gb = 0.0
local_scheduler = LocalSearchScheduler(
    config, targets, begin, end, max_iterations=1500, time_limit=300, seed=1
)
local_plan = local_scheduler.schedule()
local = execute(local_plan)

print(
    f"Merit-weighted science: {local_scheduler.initial_score[0] / 3600:.0f} -> "
    f"{local_scheduler.score[0] / 3600:.0f} merit-hours "
    f"({local_scheduler.iterations} changes tried, {local_scheduler.accepted} kept)"
)
science = [e for e in local_plan if e.obstype == ObsType.AT]
print(
    f"Local search:  {len(science):3d} snapshots, "
    f"{sum(e.obsid in bright_ids for e in science):2d} of bright targets, "
    f"{sum(e.exposure for e in science) / 3600:5.2f} h of science"
)
report("Local search", local)

fig, ax = plt.subplots(figsize=(10, 3.5))
plot_recorder(ax, limited, "PriorityScheduler", "tab:blue")
plot_recorder(ax, local, "LocalSearchScheduler", "tab:green")
shade_contacts(ax, local_plan)
ax.axhline(
    config.recorder.planning_limit_gb,
    color="tab:blue",
    ls="--",
    lw=1,
    label="planning limit",
)
ax.set_xlim(0, 24)
ax.set_ylim(0, CAPACITY_GB * 1.05)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Recorder volume (Gb)")
ax.legend(ncol=2, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## Step 10: CP-SAT With the Recorder in the Model

`CpSatScheduler` has the recorder in its constraint model: each candidate
snapshot's data goes on the recorder in the segment where its collection starts,
each segment's peak is kept within the limit, and what a contact cannot remove
carries over to the segments after it. The solver therefore chooses which
snapshots to take, and when, with the data each one costs weighed against what it
is worth, rather than finding out afterwards that one did not fit. It starts from
the priority-first plan and, like local search, never returns a worse one; it needs
the `cpsat` extra (OR-Tools).

In [ ]:
from conops.schedulers import CpSatScheduler

config.recorder.current_volume_gb = 0.0
cpsat_scheduler = CpSatScheduler(
    config,
    targets,
    begin,
    end,
    solver_time_limit=60.0,
    workers=1,
    seed=1,
    max_iterations=0,
)
cpsat_plan = cpsat_scheduler.schedule()
cpsat = execute(cpsat_plan)

print(
    f"Merit-weighted science: {cpsat_scheduler.start_score[0] / 3600:.0f} -> "
    f"{cpsat_scheduler.solver_score[0] / 3600:.0f} merit-hours; "
    f"{sum(cpsat_scheduler.solver_chunks_used)} of {len(cpsat_scheduler.solver_chunks_used)} "
    f"chunks improved, statuses {sorted(set(cpsat_scheduler.solver_statuses))}"
)
science = [e for e in cpsat_plan if e.obstype == ObsType.AT]
print(
    f"CP-SAT:        {len(science):3d} snapshots, "
    f"{sum(e.obsid in bright_ids for e in science):2d} of bright targets, "
    f"{sum(e.exposure for e in science) / 3600:5.2f} h of science"
)
report("CP-SAT", cpsat)

fig, ax = plt.subplots(figsize=(10, 3.5))
plot_recorder(ax, limited, "PriorityScheduler", "tab:blue")
plot_recorder(ax, local, "LocalSearchScheduler", "tab:green")
plot_recorder(ax, cpsat, "CpSatScheduler", "tab:red")
shade_contacts(ax, cpsat_plan)
ax.axhline(
    config.recorder.planning_limit_gb,
    color="tab:blue",
    ls="--",
    lw=1,
    label="planning limit",
)
ax.set_xlim(0, 24)
ax.set_ylim(0, CAPACITY_GB * 1.05)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Recorder volume (Gb)")
ax.legend(ncol=2, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## Step 11: Keeping Margin

The limit assumes every contact downlinks for its whole length. A contact that is
missed or cut short leaves more on the recorder than planned. Setting
`planning_limit` below the red threshold keeps margin for that: here 70%, or
22.4 Gb, which costs some of the evening's science, since the recorder has to carry
it overnight.

In [ ]:
margin_scheduler, margin_plan = plan_day(0.7, "70% limit")
margin = execute(margin_plan)
report("70% limit", margin)
config.recorder.planning_limit = 0.9

fig, ax = plt.subplots(figsize=(10, 3.5))
plot_recorder(ax, limited, "90% limit (red threshold)", "tab:blue")
plot_recorder(ax, margin, "70% limit", "tab:purple")
shade_contacts(ax, margin_plan)
ax.axhline(0.9 * CAPACITY_GB, color="tab:blue", ls="--", lw=1)
ax.axhline(0.7 * CAPACITY_GB, color="tab:purple", ls="--", lw=1)
ax.set_xlim(0, 24)
ax.set_ylim(0, CAPACITY_GB * 1.05)
ax.set_xlabel("Hours since start of day")
ax.set_ylabel("Recorder volume (Gb)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## Step 12: Replanning From the Recorder's Actual Volume

In a rolling-horizon simulation the plan is rebuilt as the day goes on, and each
new plan starts from what is actually on the recorder, so a contact that
downlinked less than planned is made up for at the next replan. Here the day is
planned in 12-hour horizons, replanned every 6 hours, starting with 16 Gb already
on the recorder from the day before.

In [ ]:
config.recorder.current_volume_gb = 16.0
rolling = RollingHorizonDITL(
    config,
    make_targets(7),
    ephem=ephemeris,
    begin=begin,
    end=end,
    horizon=timedelta(hours=12),
    replan_interval=timedelta(hours=6),
    commit_lead_time=timedelta(minutes=15),
)
rolling.calc()
config.recorder.current_volume_gb = 0.0

report("Rolling", rolling, initial_gb=16.0)
fig, ax = plt.subplots(figsize=(10, 3.5))
plot_recorder(ax, rolling, "Rolling horizon", "tab:brown")
shade_contacts(ax, rolling.plan)
for replan in rolling.replans:
    ax.axvline((replan.utime - begin.timestamp()) / 3600, color="grey", lw=0.8, ls=":")
ax.axhline(
    config.recorder.planning_limit_gb,
    color="tab:blue",
    ls="--",
    lw=1,
    label="planning limit",
)
ax.set_xlim(0, 24)
ax.set_ylim(0, CAPACITY_GB * 1.05)
ax.set_xlabel("Hours since start of day (dotted lines: replans)")
ax.set_ylabel("Recorder volume (Gb)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## Summary

* Every scheduler keeps the recorder within `recorder.planning_limit`, by default
  the red threshold, assuming every contact downlinks for its whole length. Set it
  lower for margin, or to `None` to plan without regard to the recorder.
* What a snapshot generates comes from the data model: each instrument's rate and
  per-observation amount, or the target's own `science_data` for that instrument,
  plus housekeeping from the bus and instruments at all times.
* Snapshots are never shortened to fit. The priority-first scheduler places
  higher-merit requests first, so when data runs short the lowest-merit science
  waits for the next contact; local search trades snapshots in and out on
  merit-weighted science within the same limit; and CP-SAT has the recorder in
  its model, so the data a snapshot costs is weighed against what it is worth in
  the solver itself.
* Rolling-horizon replans start from the recorder's actual volume.

See the planning guide (Planning within the recorder) and the data management
guide for details.